# Playcadia — US marketplace copy (English)

Reads `games.csv`, calls OpenAI for each **pending** title, appends valid rows to `results.csv`. At the end writes `pending.csv` and a compact report. Designed for large runs (~40k+): minimal console output; errors append to `enrich_errors.log`.

In [ ]:
from __future__ import annotations

import csv
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from openai import OpenAI

# --- Config ---
MODEL = "gpt-4o-mini"
MIN_DESCRIPTION_LEN = 20
SLEEP_SECONDS = 0.5
FORCE_REPROCESS = False  # True = ignore results.csv and reprocess all

# Large runs: avoid flooding the notebook
PROGRESS_EVERY = 250  # print one progress line every N rows processed in this run
PRINT_EACH_ERROR = False  # if True, print every API/validation error (very noisy)
MAX_ERRORS_TO_PRINT_AT_END = 40  # summary only


def resolve_data_dir() -> Path:
    """Folder containing games.csv / results.csv. Works with cwd = repo root or Playcadia/."""
    cwd = Path.cwd().resolve()
    if (cwd / "games.csv").exists():
        return cwd
    if cwd.name == "Playcadia" and (cwd / "games.csv").exists():
        return cwd
    playcadia = cwd / "Playcadia"
    if (playcadia / "games.csv").exists():
        return playcadia
    return playcadia


DATA_DIR = resolve_data_dir()
GAMES_CSV = DATA_DIR / "games.csv"
RESULTS_CSV = DATA_DIR / "results.csv"
PENDING_CSV = DATA_DIR / "pending.csv"
ERROR_LOG = DATA_DIR / "enrich_errors.log"

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Set environment variable OPENAI_API_KEY before running.")

print(f"DATA_DIR={DATA_DIR}  games.csv exists={GAMES_CSV.exists()}")

In [ ]:
EXPECTED_GAME_COLS = ["id", "console-name", "product-name"]
RESULT_COLS = ["id", "description", "recommended_age_group", "game_style"]


def load_games(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Place games.csv under {DATA_DIR}.")
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = df.columns.str.strip()
    missing = [c for c in EXPECTED_GAME_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"games.csv missing columns {missing}. Found: {list(df.columns)}")
    df = df[EXPECTED_GAME_COLS].copy()
    df = df.dropna(subset=["id"])
    df["id"] = df["id"].astype(str).str.strip()
    df = df[df["id"] != ""]
    df = df[df["id"].str.lower() != "nan"]
    df["console-name"] = df["console-name"].astype(str).str.strip()
    df["product-name"] = df["product-name"].astype(str).str.strip()
    df = df.drop_duplicates(subset=["id"], keep="first")
    return df


def row_output_is_valid(description, age, style) -> bool:
    if not isinstance(description, str) or not isinstance(age, str) or not isinstance(style, str):
        return False
    d, a, s = description.strip(), age.strip(), style.strip()
    if len(d) < MIN_DESCRIPTION_LEN or not a or not s:
        return False
    return True


def load_completed_ids_from_results(path: Path) -> set[str]:
    """Vectorized scan of results.csv — faster for tens of thousands of rows."""
    if not path.exists() or FORCE_REPROCESS:
        return set()
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = df.columns.str.strip()
    missing = [c for c in RESULT_COLS if c not in df.columns]
    if missing:
        return set()
    for c in ("description", "recommended_age_group", "game_style"):
        df[c] = df[c].fillna("").map(lambda x: str(x).strip() if pd.notna(x) else "")
    df["id"] = df["id"].fillna("").astype(str).str.strip()
    ok = (
        (df["description"].str.len() >= MIN_DESCRIPTION_LEN)
        & (df["recommended_age_group"].str.len() > 0)
        & (df["game_style"].str.len() > 0)
    )
    return set(df.loc[ok, "id"].tolist())


games_df = load_games(GAMES_CSV)
completed_ids = load_completed_ids_from_results(RESULTS_CSV)
pending_mask = ~games_df["id"].isin(completed_ids)
pending_df = games_df.loc[pending_mask].copy()

n_games = len(games_df)
n_done = len(completed_ids & set(games_df["id"]))
n_pending = len(pending_df)
print(
    f"games={n_games}  already_in_results={n_done}  to_process_this_run={n_pending}"
)

In [ ]:
SYSTEM_PROMPT = """You write catalog copy for a US marketplace that sells used video games.
Output must help shoppers quickly understand what the game is: clear, factual, neutral tone (no hype, no long plot spoilers).

Return a single valid JSON object (no markdown, no text before or after) with exactly these keys:
- "description": 2–4 short sentences in **English (US)** about gameplay and what the player does; no emojis.
- "recommended_age_group": a short label consistent with common US ratings when applicable (examples: "E", "E10+", "T", "M"; or "PEG 7" if clearly EU-sourced stock — pick the most reasonable label for the title).
- "game_style": one or two short English tags separated by commas (e.g. "2D platformer", "turn-based JRPG", "open-world action").

Use plain UTF-8 characters only; avoid curly quotes and odd symbols."""


def build_user_message(console_name: str, product_name: str) -> str:
    return (
        f"Platform (reference): {console_name}\n"
        f"Product name: {product_name}\n\n"
        "Generate the JSON listing fields for this title."
    )


def parse_and_validate(content: str) -> dict | None:
    try:
        data = json.loads(content)
    except json.JSONDecodeError:
        return None
    if not isinstance(data, dict):
        return None
    desc = data.get("description")
    age = data.get("recommended_age_group")
    style = data.get("game_style")
    if not row_output_is_valid(desc, age, style):
        return None
    return {
        "description": desc.strip(),
        "recommended_age_group": age.strip(),
        "game_style": style.strip(),
    }


client = OpenAI(api_key=api_key)


def fetch_enrichment(console_name: str, product_name: str) -> dict | None:
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_message(console_name, product_name)},
        ],
        response_format={"type": "json_object"},
        temperature=0.35,
    )
    text = completion.choices[0].message.content
    if not text:
        return None
    return parse_and_validate(text)


def append_result_row(path: Path, row: dict) -> None:
    file_exists = path.exists() and path.stat().st_size > 0
    with path.open("a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=RESULT_COLS)
        if not file_exists:
            w.writeheader()
        w.writerow(row)


def append_error_log(gid: str, message: str) -> None:
    ts = datetime.now(timezone.utc).isoformat()
    line = f"{ts}\t{gid}\t{message}\n"
    with ERROR_LOG.open("a", encoding="utf-8") as f:
        f.write(line)

In [ ]:
new_ok = 0
new_fail = 0
errors: list[tuple[str, str]] = []

total_this_run = len(pending_df)
processed = 0

if total_this_run == 0:
    print("Nothing to process — all IDs already have valid rows in results.csv.")
else:
    print(
        f"Starting: {total_this_run} rows (progress line every {PROGRESS_EVERY})"
    )

for gid, console, product in zip(
    pending_df["id"],
    pending_df["console-name"],
    pending_df["product-name"],
    strict=True,
):
    processed += 1
    try:
        parsed = fetch_enrichment(console, product)
    except Exception as e:
        new_fail += 1
        msg = repr(e)
        errors.append((gid, msg))
        append_error_log(gid, msg)
        if PRINT_EACH_ERROR:
            print(f"[err] id={gid} {e}")
        time.sleep(SLEEP_SECONDS)
        if processed % PROGRESS_EVERY == 0 or processed == total_this_run:
            print(
                f"progress {processed}/{total_this_run}  ok={new_ok}  fail={new_fail}"
            )
        continue
    if parsed is None:
        new_fail += 1
        msg = "invalid JSON or empty fields"
        errors.append((gid, msg))
        append_error_log(gid, msg)
        if PRINT_EACH_ERROR:
            print(f"[skip] id={gid}")
        time.sleep(SLEEP_SECONDS)
        if processed % PROGRESS_EVERY == 0 or processed == total_this_run:
            print(
                f"progress {processed}/{total_this_run}  ok={new_ok}  fail={new_fail}"
            )
        continue
    out = {"id": gid, **parsed}
    append_result_row(RESULTS_CSV, out)
    completed_ids.add(gid)
    new_ok += 1
    time.sleep(SLEEP_SECONDS)
    if processed % PROGRESS_EVERY == 0 or processed == total_this_run:
        print(f"progress {processed}/{total_this_run}  ok={new_ok}  fail={new_fail}")

print(f"--- done this run: saved_ok={new_ok}  failed={new_fail}  (see {ERROR_LOG.name} for errors) ---")

In [ ]:
# Refresh pending.csv + report (run after the loop cell)
errors = globals().get("errors", [])

completed_ids_final = load_completed_ids_from_results(RESULTS_CSV)
still_pending = games_df[~games_df["id"].isin(completed_ids_final)].copy()
still_pending.to_csv(PENDING_CSV, index=False, encoding="utf-8-sig")

total = len(games_df)
done = len(games_df[games_df["id"].isin(completed_ids_final)])
pend = total - done
pct = (100.0 * done / total) if total else 0.0

report = {
    "total_in_games_csv": total,
    "completed_with_valid_data": done,
    "pending": pend,
    "pct_complete": round(pct, 2),
    "pending_file": str(PENDING_CSV),
    "error_log": str(ERROR_LOG),
}
print(json.dumps(report, ensure_ascii=False, indent=2))

report_path = DATA_DIR / "last_run_report.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Wrote {report_path}")

if errors:
    print(
        f"Sample errors from this run (showing up to {MAX_ERRORS_TO_PRINT_AT_END} of {len(errors)}):"
    )
    for gid, msg in errors[:MAX_ERRORS_TO_PRINT_AT_END]:
        print(f"  {gid}: {msg[:200]}")
    if len(errors) > MAX_ERRORS_TO_PRINT_AT_END:
        print(f"  ... {len(errors) - MAX_ERRORS_TO_PRINT_AT_END} more in {ERROR_LOG.name}")